# Create Conquer Cancer Foundation Awards

Creates the grants and awards of **Conquer Cancer, the ASCO Foundation** (Conquer Cancer
Foundation of the American Society of Clinical Oncology) from its own recipient database.

**Prerequisite:** `scripts/local/conquer_cancer_to_s3.py` (uploads the parquet).

**Source:** the "View All Recipients" Grant & Award database
(https://www.asco.org/career-development/grants-awards/grant-award-recipients, linked from
https://www.conquer.org/cancer-research/grants-awards), which is one static JSON file
(`https://www.asco.org/assets/contents/grant-award-recipients.json`, ladder step 0: the
funder's own bulk file). asco.org and conquer.org robots.txt: `User-agent: * / Allow: /`.

**8,838 recipients, 1984-2026** across 47 programme labels. Every programme is kept
(scope rule 2026-10-01); `funding_type` separates them:
- `career_development` (YIA 1,837 incl. NCI-funded / "fundable" variants, CDA 432, Global
  Oncology YIA 50, Nachman Junior Faculty 16) and `research` (ACRA 39, International
  Innovation Grant 64, Registry, Improving Cancer Care, Career Pathway, Research
  Professorship): titles + USD amounts.
- `fellowship` (Long-Term International Fellowship, Bonadonna, ASCO Fellowship Award) and
  `training` (IDEA 521, Oncology Summer Internship 281, Medical Student Rotation 200,
  ACORE 83).
- FLAGGED, not clearly research funding (raw column `research_pathway = false`, 5,250 rows):
  meeting Merit / abstract awards (`funding_type = 'prize'`; Annual Meeting Merit Award
  3,078, symposium merit awards, Breakthrough awards; title = the recognised abstract),
  travel awards (`travel`) and the institutional Clinical Trials Participation Award
  (`prize`, 116, practice name only, no person).

**Funder:** all rows go to Conquer Cancer F4320306239. The programme was ASCO's until the
foundation took it over (founded 1999 as The ASCO Foundation); Conquer Cancer presents the
whole 1984-present history as its own programme. Twin: American Society of Clinical
Oncology F4320308982 (separate ROR 04fy6j421) carries 47 citation stubs that are also
Project IDs here.

**Field mapping:** `display_name` = Title (research title, or abstract title for merit
awards), else `Conquer Cancer {Program} {Year}: {First Last}` (or the institution for
participation awards); `funder_scheme` = Program; `start_year` = Year (no dates published,
`start_date` NULL); `lead_investigator` = First / Last Name (given separately in the source),
affiliation = Institution as printed; participation awards carry the practice as affiliation
only. `amount` = Grant Amount ($) in USD where published (grant programmes only; merit,
training and travel awards publish no amount).

**`funder_award_id` (2.1.1):** the Project ID, e.g. `2021YIA-6088880692` (recent) or
`12822` (older): the number citing works put in Crossref funding metadata. 110 Conquer
Cancer citation stubs equal a Project ID exactly, so they collapse onto these awards. The
3 records without a Project ID get a synthetic `CCF-{programme initials}{year}-{surname}` key.

**Funder details (Path A, F4320* Crossref-registered):**
- funder_id: `4320306239`
- display_name / ror_id / doi: from `openalex.funders.funders` (ror 027327e78, doi 10.13039/100000982)

**Priority:** `577` (direct funder ingest; higher wins under the 2026-06-20 DESC dedup).

## Step 1: Create Staging Table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.conquer_cancer_raw
USING delta
AS
SELECT *, current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/conquer_cancer/conquer_cancer_projects.parquet`;

In [ ]:
%sql
SELECT COUNT(*) as total, COUNT(DISTINCT funder_award_id) as distinct_ids,
       MIN(year) as min_year, MAX(year) as max_year,
       SUM(CASE WHEN LOWER(research_pathway) = 'true' THEN 1 ELSE 0 END) as research_pathway_rows
FROM openalex.awards.conquer_cancer_raw;

In [ ]:
%sql
DESCRIBE openalex.awards.conquer_cancer_raw;

In [ ]:
%sql
SELECT program, funding_type, research_pathway, COUNT(*) AS n, COUNT(amount) AS with_amount
FROM openalex.awards.conquer_cancer_raw GROUP BY 1, 2, 3 ORDER BY n DESC;

## Step 1.6: Funder existence check (Path A)

In [ ]:
%sql
SELECT assert_true(COUNT(*) = 1, 'F4320306239 must have exactly one openalex.funders.funders row') AS funder_ok,
       MAX(display_name) AS display_name, MAX(ror_id) AS ror_id, MAX(doi) AS doi
FROM openalex.funders.funders
WHERE funder_id = 4320306239;

## Step 2: Create Conquer Cancer Awards Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.conquer_cancer_awards
USING delta
AS
WITH
src_funder AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.funders.funders
    WHERE funder_id = 4320306239  -- Conquer Cancer Foundation
),
g AS (
    SELECT *,
           TRIM(funder_award_id) AS award_key,
           NULLIF(TRIM(lead_given_name), '') AS given,
           NULLIF(TRIM(lead_family_name), '') AS family,
           NULLIF(TRIM(institution), '') AS inst
    FROM openalex.awards.conquer_cancer_raw
    WHERE funder_award_id IS NOT NULL AND TRIM(funder_award_id) != ''
)

SELECT
    abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(g.award_key)))) % 9000000000 as id,
    COALESCE(NULLIF(TRIM(g.title), ''),
             CONCAT('Conquer Cancer ', g.program, ' ', g.year, ': ',
                    COALESCE(NULLIF(CONCAT_WS(' ', g.given, g.family), ''), g.inst))) as display_name,
    CAST(NULL AS STRING) as description,
    f.funder_id,
    g.award_key as funder_award_id,
    TRY_CAST(g.amount AS DOUBLE) as amount,
    CASE WHEN TRY_CAST(g.amount AS DOUBLE) IS NOT NULL THEN 'USD' END as currency,
    struct(
        CONCAT('https://openalex.org/F', f.funder_id) as id,
        f.display_name,
        f.ror_id,
        f.doi
    ) as funder,
    g.funding_type as funding_type,
    NULLIF(TRIM(g.program), '') as funder_scheme,
    'conquer_cancer_recipients_db' as provenance,
    CAST(NULL AS DATE) as start_date,
    CAST(NULL AS DATE) as end_date,
    TRY_CAST(g.year AS INT) as start_year,
    CAST(NULL AS INT) as end_year,
    -- struct field order MUST match openalex_awards_raw: given, family, orcid, role_start, affiliation.
    CASE WHEN g.family IS NOT NULL OR g.inst IS NOT NULL THEN named_struct(
        'given_name', g.given,
        'family_name', g.family,
        'orcid', CAST(NULL AS STRING),
        'role_start', CAST(NULL AS DATE),
        'affiliation', named_struct(
            'name', g.inst,
            'country', CAST(NULL AS STRING),
            'ids', CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>)
        )
    ) END as lead_investigator,
    CAST(NULL AS STRUCT<given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
         affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>>) as co_lead_investigator,
    CAST(NULL AS ARRAY<STRUCT<given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
         affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>>>) as investigators,
    g.landing_page_url as landing_page_url,
    CAST(NULL AS STRING) as doi,
    concat('https://api.openalex.org/works?filter=awards.id:G',
           abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(g.award_key)))) % 9000000000) as works_api_url,
    current_timestamp() as created_date,
    current_timestamp() as updated_date
FROM g
CROSS JOIN src_funder f;

## Shape check (2.1.1)
Project IDs: `{yyyy}{CODE}-{10 digits}` (2016+) or short programme-prefixed / numeric ids
(older): one token of letters, digits, `_` and `-`, no spaces. The 3 records without one
carry the synthetic `CCF-` key.

In [ ]:
%sql
SELECT assert_true(SUM(CASE WHEN funder_award_id RLIKE '^[A-Za-z0-9][A-Za-z0-9_-]{1,40}$' THEN 0 ELSE 1 END) = 0,
                   'unexpected funder_award_id shape') AS shape_ok,
       SUM(CASE WHEN funder_award_id LIKE 'CCF-%' THEN 1 ELSE 0 END) AS synthetic_keys
FROM openalex.awards.conquer_cancer_awards;

In [ ]:
%sql
SELECT assert_true(COUNT(*) = COUNT(DISTINCT id), 'duplicate award ids') AS unique_ok, COUNT(*) AS n
FROM openalex.awards.conquer_cancer_awards;

## Step 3: Insert into openalex_awards_raw

In [ ]:
%sql
-- Remove previous data for this source before inserting fresh data.
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'conquer_cancer_recipients_db' AND priority = 577;

-- Priority 577: direct-from-funder ingest of the foundation's own recipient database.
-- Higher wins under the 2026-06-20 DESC dedup (oxjob #500).
INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id, amount, currency,
    funder, funding_type, funder_scheme, provenance, start_date, end_date,
    start_year, end_year, lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url, created_date, updated_date,
    577 as priority
FROM openalex.awards.conquer_cancer_awards;

## Verification Queries

In [ ]:
%sql
SELECT id, display_name, funder_award_id, funder_scheme, funding_type, amount, start_year,
       lead_investigator.given_name, lead_investigator.family_name,
       lead_investigator.affiliation.name
FROM openalex.awards.conquer_cancer_awards LIMIT 20;

In [ ]:
%sql
SELECT funding_type, funder_scheme, COUNT(*) as cnt, COUNT(amount) as with_amount
FROM openalex.awards.conquer_cancer_awards GROUP BY 1, 2 ORDER BY cnt DESC;

In [ ]:
%sql
SELECT start_year, COUNT(*) as cnt
FROM openalex.awards.conquer_cancer_awards
WHERE start_year IS NOT NULL GROUP BY start_year ORDER BY start_year;

In [ ]:
%sql
-- 6.4a frequency check: PI top-20 must be a real long-tail.
SELECT lead_investigator.given_name AS given, lead_investigator.family_name AS family, COUNT(*) AS n
FROM openalex.awards.conquer_cancer_awards
GROUP BY 1, 2 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
SELECT display_name, COUNT(*) AS n
FROM openalex.awards.conquer_cancer_awards
GROUP BY 1 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
-- 6.3 / 6.7 coverage.
SELECT
    COUNT(*) as total,
    COUNT(display_name) as has_title,
    COUNT(amount) as has_amount,
    COUNT(DISTINCT currency) as distinct_currencies,
    ROUND(MIN(amount), 0) as min_amt,
    ROUND(MAX(amount), 0) as max_amt,
    ROUND(AVG(amount), 0) as avg_amt,
    COUNT(start_year) as has_start_year,
    COUNT(lead_investigator.family_name) as has_pi
FROM openalex.awards.conquer_cancer_awards;

In [ ]:
%sql
-- 2.1.1: acknowledgement shells whose award id equals a published Project ID (they collapse onto these awards).
SELECT r.provenance, COUNT(*) AS shells, COUNT(a.id) AS matching_project_ids
FROM openalex.awards.openalex_awards_raw r
LEFT JOIN openalex.awards.conquer_cancer_awards a ON a.id = r.id
WHERE r.funder_id = 4320306239 AND r.priority < 3
GROUP BY 1;

In [ ]:
%sql
-- Confirm rows reached the shared raw table at the assigned priority (6.8).
SELECT provenance, priority, COUNT(*) as n
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'conquer_cancer_recipients_db'
GROUP BY provenance, priority;